# 03 — 标签准备
**任务：下载NSW SEED 2024年4月裸土侵蚀数据 → 裁切 → 重投影配准 → 保存训练标签**

标签数据：月度裸土侵蚀量（t/ha/月）2024年4月
目标：配准到 Sentinel-2 特征矩阵坐标系（EPSG:32755，10m，16811×18790）

## 1. 导入包

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.windows import from_bounds
import requests
import subprocess

from src.data_utils import load_config, ensure_dir

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('所有包导入成功')

所有包导入成功


## 2. 加载配置

In [2]:
cfg = load_config('../configs/config.yaml')

bbox          = cfg['region']['bbox_test']
BBOX          = [bbox['lon_min'], bbox['lat_min'], bbox['lon_max'], bbox['lat_max']]
LABEL         = '2024_04'

label_dir     = ensure_dir('../data/raw/labels')
processed_dir = ensure_dir('../data/processed')
fig_dir       = ensure_dir('../results/figures')

# 目标参数：与 Sentinel-2 2024年4月特征矩阵完全一致
FEATURE_TIF   = processed_dir / f'NSW_test_NDVI_composite_{LABEL}.tif'

with rasterio.open(FEATURE_TIF) as feat:
    TARGET_CRS       = feat.crs
    TARGET_TRANSFORM = feat.transform
    TARGET_HEIGHT    = feat.height
    TARGET_WIDTH     = feat.width

print(f'测试区 BBOX     : {BBOX}')
print(f'目标坐标系      : {TARGET_CRS}')
print(f'目标分辨率      : {TARGET_TRANSFORM.a:.1f} m')
print(f'目标形状        : {TARGET_HEIGHT} × {TARGET_WIDTH}')

测试区 BBOX     : [147.0, -34.0, 149.0, -32.5]
目标坐标系      : EPSG:32755
目标分辨率      : 10.0 m
目标形状        : 16811 × 18790


## 3. 下载 2024年4月裸土侵蚀标签

In [3]:
URL      = 'https://datasets.seed.nsw.gov.au/dataset/db9a8486-7401-4213-99b8-66037fda1a9c/resource/3b7135a2-6afd-4dee-8397-61da90a9d87f/download/ero_202404b.tif'
raw_path = label_dir / 'ero_202404b.tif'

if raw_path.exists():
    print(f'文件已存在，跳过下载: {raw_path.name}')
else:
    print('正在下载...')
    response = requests.get(URL, stream=True)
    total    = int(response.headers.get('content-length', 0))
    with open(raw_path, 'wb') as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=1024*1024):
            f.write(chunk)
            downloaded += len(chunk)
            print(f'\r进度: {downloaded/1024/1024:.1f} / {total/1024/1024:.1f} MB', end='')
    print(f'\n下载完成')

print(f'文件大小: {raw_path.stat().st_size/1024/1024:.1f} MB')

文件已存在，跳过下载: ero_202404b.tif
文件大小: 336.2 MB


## 4. 查看原始标签信息

In [4]:
with rasterio.open(raw_path) as src:
    print(f'坐标系  : {src.crs}')
    print(f'分辨率  : {src.res}')
    print(f'形状    : {src.width} × {src.height}')
    print(f'NoData  : {src.nodata}')
    print(f'范围    : {src.bounds}')
    raw_nodata = src.nodata

坐标系  : EPSG:4283
分辨率  : (0.0010000000000000005, 0.001)
形状    : 14525 × 9843
NoData  : -3.4028230607370965e+38
范围    : BoundingBox(left=140.10290697138387, bottom=-37.75622774602686, right=154.62790697138388, top=-27.91322774602686)


## 5. 裁切到测试区范围

In [5]:
clipped_path = processed_dir / f'label_ero_202404_clipped.tif'

with rasterio.open(raw_path) as src:
    window           = from_bounds(
        left=BBOX[0], bottom=BBOX[1],
        right=BBOX[2], top=BBOX[3],
        transform=src.transform
    )
    clipped          = src.read(1, window=window).astype('float32')
    clipped_transform = src.window_transform(window)
    clipped_crs      = src.crs

    # 替换 NoData
    clipped[clipped == raw_nodata] = np.nan

    with rasterio.open(
        clipped_path, 'w', driver='GTiff',
        height=clipped.shape[0], width=clipped.shape[1],
        count=1, dtype='float32',
        crs=clipped_crs, transform=clipped_transform,
        nodata=np.nan
    ) as dst:
        dst.write(clipped, 1)

print(f'裁切完成: {clipped_path.name}')
print(f'裁切后形状    : {clipped.shape}')
print(f'有效像素比例  : {np.isfinite(clipped).mean()*100:.1f}%')
print(f'数值范围      : {np.nanmin(clipped):.4f} ~ {np.nanmax(clipped):.4f} t/ha/月')

裁切完成: label_ero_202404_clipped.tif
裁切后形状    : (1500, 2000)
有效像素比例  : 100.0%
数值范围      : 0.0409 ~ 641.3892 t/ha/月


## 6. 重投影配准到 Sentinel-2 坐标系

In [6]:
import gc
gc.collect()

reproj_path  = processed_dir / f'label_ero_202404_reproj.tif'
label_reproj = np.full((TARGET_HEIGHT, TARGET_WIDTH), np.nan, dtype='float32')

with rasterio.open(clipped_path) as src:
    reproject(
        source=rasterio.band(src, 1),
        destination=label_reproj,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=TARGET_TRANSFORM,
        dst_crs=TARGET_CRS,
        resampling=Resampling.bilinear,
        src_nodata=np.nan,
        dst_nodata=np.nan
    )

# 保存重投影结果
with rasterio.open(
    reproj_path, 'w', driver='GTiff',
    height=TARGET_HEIGHT, width=TARGET_WIDTH,
    count=1, dtype='float32',
    crs=TARGET_CRS, transform=TARGET_TRANSFORM,
    nodata=np.nan, compress='lzw'
) as dst:
    dst.write(label_reproj, 1)

print(f'重投影完成: {reproj_path.name}')
print(f'形状      : {label_reproj.shape}')
print(f'有效像素  : {np.isfinite(label_reproj).mean()*100:.1f}%')
print(f'数值范围  : {np.nanmin(label_reproj):.4f} ~ {np.nanmax(label_reproj):.4f} t/ha/月')

重投影完成: label_ero_202404_reproj.tif
形状      : (16811, 18790)
有效像素  : 98.1%
数值范围  : 0.0414 ~ 637.3094 t/ha/月


## 7. 可视化特征与标签对比

In [ ]:
feature_ndvi = np.load(processed_dir / f'NSW_test_features_{LABEL}.npy')[:,:,0]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('特征与标签对比\n2024年4月 Sentinel-2 NDVI vs 2024年4月裸土侵蚀量', fontsize=13)

im0 = axes[0].imshow(feature_ndvi, cmap='RdYlGn', vmin=-0.2, vmax=0.8)
axes[0].set_title('NDVI（特征）')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(label_reproj, cmap='YlOrRd', vmin=0, vmax=50)
axes[1].set_title('裸土侵蚀量 t/ha/月（标签）')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.tight_layout()
fig_path = fig_dir / 'NSW_feature_vs_label_2024_04.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'图像已保存: {fig_path.name}')
plt.show()

## 8. 保存训练标签并验证对齐

In [ ]:
label_npy_path = processed_dir / f'NSW_test_label_ero_{LABEL.replace("_","")}.npy'
np.save(label_npy_path, label_reproj)

features = np.load(processed_dir / f'NSW_test_features_{LABEL}.npy')

print(f'标签已保存: {label_npy_path.name}')
print(f'标签形状  : {label_reproj.shape}')
print(f'特征形状  : {features.shape}')

assert label_reproj.shape == features.shape[:2], '形状不匹配！'
print('形状对齐  : ✓')
print(f'\n训练数据摘要:')
print(f'  特征: {features.shape}  (H × W × 7通道)')
print(f'  标签: {label_reproj.shape}  (H × W)')
print(f'  标签范围: {np.nanmin(label_reproj):.4f} ~ {np.nanmax(label_reproj):.4f} t/ha/月')
print(f'  有效像素: {np.isfinite(label_reproj).mean()*100:.1f}%')

## 9. 提交到 GitHub

In [ ]:
commands = [
    ['git', 'add', 'notebooks/03_label_preparation.ipynb'],
    ['git', 'add', 'results/figures/NSW_feature_vs_label_2024_04.png'],
    ['git', 'commit', '-m', 'feat: notebook 03 refactor - label aligned to 2024-04 Sentinel-2 features'],
    ['git', 'push']
]

for cmd in commands:
    result = subprocess.run(cmd, cwd='..', capture_output=True, text=True)
    print(' '.join(cmd))
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)